# TCA Demo: KDB-X → Parquet

Generates all four tables needed for Transaction Cost Analysis — `trades`, `quotes`, `orders`, `executions` — using the `di.simtick` and `di.simorder` KDB-X modules, visualizes good vs. bad execution against the market, then exports everything to Parquet for ClickHouse ingestion.

Runs the `good` and `bad` presets from `di.simorder` against the same market data, combined into shared `orders`/`executions` tables — so the exported files already contain a ready side-by-side comparison.

## Import Libraries

In [ ]:
import os
import pykx as kx
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Get paths
module_path = os.path.expanduser('~/kdbx-modules')
tick_presets_path = f"{module_path}/di/simtick/presets.csv"
order_presets_path = f"{module_path}/di/simorder/presets.csv"

# Set QPATH and load modules
kx.q(f'setenv[`QPATH;"{module_path}"]')
kx.q('simtick:use`di.simtick')
kx.q('simorder:use`di.simorder')
kx.q(f'tickcfgs:simtick.loadconfig`$":{tick_presets_path}"')
kx.q(f'ordcfgs:simorder.loadconfig`$":{order_presets_path}"')

print("Modules loaded successfully!")

## Available presets

In [ ]:
print("Market presets (di.simtick):")
display(kx.q('key tickcfgs').pd())
print("\nOrder presets (di.simorder):")
display(kx.q('key ordcfgs').pd())

## Generate market data

`generatequotes` defaults to `1b` in `presets.csv`, so both `trades` and `quotes` are produced.

In [ ]:
scenario = 'default'  # Change to: 'volatile', 'jumpy'

kx.q(f'result:simtick.run[tickcfgs`{scenario}]')

trades = kx.q('result`trade').pd()
quotes = kx.q('result`quote').pd()

print(f'Trades: {len(trades):,}   Quotes: {len(quotes):,}')
trades.head(3)

## Generate orders + executions (good vs. bad)

Runs `di.simorder` twice — `good` (even pacing, tight spread capture) and `bad` (frontloaded pacing, wide spread capture) — against the same market data.

In [ ]:
kx.q('goodresult:simorder.run[ordcfgs`good;result`trade;result`quote]')
kx.q('badresult:simorder.run[ordcfgs`bad;result`trade;result`quote]')

# Combine into single orders/executions tables
kx.q('orders:goodresult[`order],badresult[`order]')
kx.q('executions:goodresult[`executions],badresult[`executions]')

orders = kx.q('orders').pd()
executions = kx.q('executions').pd()

print(f'Orders: {len(orders)}   Executions: {len(executions)}')

# Sanity check: quantities must sum exactly to orderqty for each order
qty_check = executions.groupby('orderid')['qty'].sum().reset_index().merge(
    orders[['orderid', 'orderqty']], on='orderid'
)
if (qty_check['qty'] == qty_check['orderqty']).all():
    print('✓  All execution quantities sum exactly to orderqty')
else:
    print('⚠  Quantity mismatch detected:')
    display(qty_check)

orders

## VWAP & implementation shortfall

Quick sanity check before export — not the full TCA analysis (that happens in ClickHouse).

In [ ]:
summary = (
    executions.groupby('orderid')
    .apply(lambda g: (g['price'] * g['qty']).sum() / g['qty'].sum(), include_groups=False)
    .rename('vwap')
    .reset_index()
    .merge(orders[['orderid', 'side', 'arrivalprice']], on='orderid')
)
summary['shortfall_bps'] = 10000 * (summary['vwap'] - summary['arrivalprice']) / summary['arrivalprice']
summary

## Visualize: good vs. bad execution against the market

Quotes (bid/ask spread), market trades, and both orders' executions overlaid on the same time window — the visual version of the TCA story: good execution (green) stays close to mid and spread out in time, bad execution (red) clusters early and crosses toward the far touch.

In [ ]:
# Window around the order's execution horizon
ts1 = orders['starttime'].min() - pd.Timedelta(seconds=30)
ts2 = orders['endtime'].max() + pd.Timedelta(seconds=30)

t_mask = (trades['time'] >= ts1) & (trades['time'] <= ts2)
q_mask = (quotes['time'] >= ts1) & (quotes['time'] <= ts2)

t = trades[t_mask].copy()
q = quotes[q_mask].copy()

good_ex = executions[executions['orderid'] == orders.loc[0, 'orderid']]
bad_ex = executions[executions['orderid'] == orders.loc[1, 'orderid']]

fig, ax = plt.subplots(figsize=(14, 6))

# Bid / Ask as step lines
ax.step(q['time'], q['ask'], where='post', color='#e05c5c', linewidth=1.0, label='Ask', zorder=2)
ax.step(q['time'], q['bid'], where='post', color='#5c8ae0', linewidth=1.0, label='Bid', zorder=2)
ax.fill_between(q['time'], q['bid'], q['ask'], step='post', alpha=0.10, color='grey', label='Spread')

# Background market trades
ax.scatter(t['time'], t['price'], c='lightgrey', s=10, zorder=1, label='Market trade')

# Good vs bad executions
ax.scatter(good_ex['time'], good_ex['price'], c='#2ca02c', s=good_ex['qty'] / 5,
           zorder=4, label='Good execution', edgecolors='black', linewidths=0.5)
ax.scatter(bad_ex['time'], bad_ex['price'], c='#d62728', s=bad_ex['qty'] / 5,
           zorder=4, label='Bad execution', edgecolors='black', linewidths=0.5)

# Arrival price reference line
arrival = orders.loc[0, 'arrivalprice']
ax.axhline(arrival, color='black', linestyle='--', linewidth=1, alpha=0.6, label=f'Arrival price ({arrival:.2f})')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=30, ha='right')

ax.set_title('Good vs. Bad Execution Against the Market', fontsize=13)
ax.set_xlabel('Time')
ax.set_ylabel('Price ($)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

print(f"Good execution: {len(good_ex)} fills, VWAP {summary.loc[0,'vwap']:.4f}, shortfall {summary.loc[0,'shortfall_bps']:.2f} bps")
print(f"Bad  execution: {len(bad_ex)} fills, VWAP {summary.loc[1,'vwap']:.4f}, shortfall {summary.loc[1,'shortfall_bps']:.2f} bps")

## Export all four tables to Parquet

Direct kdb → pyarrow → Parquet via PyKX's `.pa()` conversion. No CSV intermediate, no precision loss on nanosecond timestamps.

In [ ]:
output_dir = Path(module_path) / 'data'
output_dir.mkdir(exist_ok=True)

table_names = ['trades', 'quotes', 'orders', 'executions']
q_names = {'trades': 'result`trade', 'quotes': 'result`quote', 'orders': 'orders', 'executions': 'executions'}

for name in table_names:
    tbl = kx.q(q_names[name])
    path = output_dir / f'{name}.parquet'
    pq.write_table(tbl.pa(), path)
    size_kb = path.stat().st_size / 1024
    print(f'{name}.parquet written ({size_kb:.1f} KB)')

## Verify

Read each file back and confirm shape/schema before handing off for ClickHouse ingestion.

In [ ]:
for name in table_names:
    path = output_dir / f'{name}.parquet'
    t = pq.read_table(path)
    print(f'--- {name} ---')
    print(f'rows: {t.num_rows}, columns: {t.column_names}')
    display(t.to_pandas().head(3))
    print()